# Spatio-Temporal LSTM & Transformer — Univariate Training Pipeline

End-to-end training on a **single flow variable** (u-velocity).

| Parameter | Value |
|-----------|-------|
| Input features  | **3** — `[u, x_norm, y_norm]` |
| Output channels | **1** — `u` |
| kNN spatial graph | **optional** — `USE_KNN = True / False` |

## 1 — Configuration

Edit the settings below before running.

In [ ]:
# ============================================================
#  USER SETTINGS
# ============================================================

# GPU
USE_MULTI_GPU       = True
AUTO_SELECT_DEVICES = True
DEVICE_IDS          = [0, 1]

USE_MOCK_DATA       = True   # False -> load U_VELOCITY_CSV

# Model
MODEL_TYPE = "SpatioTemporalTransformer"  # or "SpatioTemporalLSTM"

# Data path
U_VELOCITY_CSV = "/kaggle/input/your-dataset/master_u_velocity.csv"

# Spatial graph (optional)
USE_KNN     = False  # True -> k-nearest-neighbour spatial encoder
K_NEIGHBORS = 10    # number of spatial neighbours per cell (ignored when USE_KNN=False)

# Temporal windows
INPUT_SEQ_LENGTH  = 15
PRED_FULL_HORIZON = 30

# Architecture
SPATIAL_EMBED_DIM = 64
HIDDEN_SIZE       = 256
NUM_LAYERS        = 3
NHEAD             = 8
DROPOUT           = 0.2

# Training
BATCH_SIZE      = 128
LEARNING_RATE   = 1e-3
WEIGHT_DECAY    = 1e-4
EPOCHS          = 50
PATIENCE        = 10
GRAD_CLIP       = 1.0
SCHEDULER_STEP  = 15
SCHEDULER_GAMMA = 0.5

# Loss
LAMBDA_SPATIAL_SMOOTH = 0.1

# Split
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15

# Performance
USE_AMP             = True
PRELOAD_TO_GPU      = False
CACHE_NEIGHBORS     = False
CACHE_VRAM_FRACTION = 0.70
NUM_WORKERS         = 2
PIN_MEMORY          = True

# Output
OUTPUT_DIR     = "outputs_univariate"
CHECKPOINT_DIR = "checkpoints_univariate"

## 2 — Imports & Device Setup

In [ ]:
from __future__ import annotations
import math, os, time, warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

warnings.filterwarnings('ignore')
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name}  ({props.total_memory / 1024**3:.1f} GB VRAM)')

### Device setup

In [ ]:
def setup_devices(use_multi_gpu, auto_select, device_ids):
    if not torch.cuda.is_available():
        print('WARNING: CUDA not available — falling back to CPU.')
        cpu = torch.device('cpu')
        return cpu, [cpu]
    if not use_multi_gpu:
        dev = torch.device('cuda:0')
        print(f'Single-GPU mode: {torch.cuda.get_device_name(0)}')
        return dev, [dev]
    ids = list(range(torch.cuda.device_count())) if auto_select else [
        i for i in device_ids if i < torch.cuda.device_count()
    ]
    if not ids:
        dev = torch.device('cuda:0')
        return dev, [dev]
    devices = [torch.device(f'cuda:{i}') for i in ids]
    if len(devices) == 1:
        print(f'Single GPU: {torch.cuda.get_device_name(ids[0])}')
    else:
        print(f'Multi-GPU mode: {len(devices)} GPUs')
        for i, d in zip(ids, devices):
            gb = torch.cuda.get_device_properties(i).total_memory / 1024**3
            print(f'  {d}  {torch.cuda.get_device_name(i)}  ({gb:.1f} GB)')
    return devices[0], devices


PRIMARY_DEVICE, ALL_DEVICES = setup_devices(USE_MULTI_GPU, AUTO_SELECT_DEVICES, DEVICE_IDS)
IS_MULTI_GPU = len(ALL_DEVICES) > 1
print(f'\nPrimary device : {PRIMARY_DEVICE}')
print(f'Multi-GPU mode : {IS_MULTI_GPU}  ({len(ALL_DEVICES)} device(s))')

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

os.makedirs(OUTPUT_DIR,     exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 3 — Data Pipeline

Loads the u-velocity CSV, normalises, appends spatial coordinates as static
features, and optionally builds a kNN spatial graph.

In [ ]:
# ------------------------------------------------------------------
# 3.1  Load raw CSV (u-velocity only)
# ------------------------------------------------------------------

def load_csv_data_univariate(u_velocity_csv, chunk_size=None):
    """
    Load the u-velocity CSV file.

    CSV format:
      - First two columns : x, y  spatial coordinates
      - Remaining columns : timestep values

    Returns
    -------
    data   : np.ndarray  shape (n_cells, n_timesteps, 1)
    coords : np.ndarray  shape (n_cells, 2)
    """
    print(f'  Loading: {u_velocity_csv}')
    df = pd.read_csv(u_velocity_csv)
    if chunk_size is not None:
        df = df.iloc[:chunk_size]
    coords = df.iloc[:, :2].values.astype(np.float32)
    u_vals = df.iloc[:, 2:].values.astype(np.float32)
    data   = u_vals[:, :, np.newaxis]   # (n_cells, n_t, 1)
    print(f'  Data shape   : {data.shape}  (cells x timesteps x 1)')
    print(f'  Coords shape : {coords.shape}')
    return data, coords


# ------------------------------------------------------------------
# 3.2  Normalisation
# ------------------------------------------------------------------

def normalize_data(data):
    """
    Z-score normalise the u-velocity channel.

    Returns
    -------
    data_norm : np.ndarray  shape (n_cells, n_t, 1)
    scaler    : fitted StandardScaler
    """
    n_cells, n_t, _ = data.shape
    flat   = data[:, :, 0].reshape(-1, 1)
    scaler = StandardScaler()
    flat_n = scaler.fit_transform(flat)
    data_norm = flat_n.reshape(n_cells, n_t, 1).astype(np.float32)
    print(f'  u_velocity: mean={scaler.mean_[0]:.4f}, std={scaler.scale_[0]:.4f}')
    return data_norm, scaler


# ------------------------------------------------------------------
# 3.3  Build feature tensor  [u_norm, x_norm, y_norm]
# ------------------------------------------------------------------

def build_feature_tensor(data_norm, coords):
    """
    Append normalised spatial coordinates as static features.

    Returns
    -------
    features : np.ndarray  shape (n_cells, n_timesteps, 3)
               Channel order: [u_norm, x_norm, y_norm]
    """
    n_cells, n_t, _ = data_norm.shape
    coords_min  = coords.min(axis=0, keepdims=True)
    coords_max  = coords.max(axis=0, keepdims=True)
    coords_norm = (coords - coords_min) / (coords_max - coords_min + 1e-8)
    x_feat = np.tile(coords_norm[:, 0:1, np.newaxis], (1, n_t, 1))  # (n_cells,n_t,1)
    y_feat = np.tile(coords_norm[:, 1:2, np.newaxis], (1, n_t, 1))  # (n_cells,n_t,1)
    features = np.concatenate([data_norm, x_feat, y_feat], axis=-1).astype(np.float32)
    print(f'Feature tensor shape: {features.shape}  [u_norm, x_norm, y_norm]')
    return features


def validate_data(data):
    if np.isnan(data).any():
        print('WARNING: NaN values detected in data!')
        return False
    if np.isinf(data).any():
        print('WARNING: Inf values detected in data!')
        return False
    print(f'  Data range: [{data.min():.4f}, {data.max():.4f}]')
    print('  Data validation passed.')
    return True


# ------------------------------------------------------------------
# 3.4  Optional kNN spatial graph
# ------------------------------------------------------------------

def build_knn_graph(coords, k):
    """
    Build a k-nearest-neighbour spatial graph.

    Returns
    -------
    neighbor_indices   : np.ndarray  shape (n_cells, k)
    neighbor_distances : np.ndarray  shape (n_cells, k)
    """
    print(f'Building kNN graph: k={k}, n_cells={coords.shape[0]} ...')
    tree = cKDTree(coords)
    distances, indices = tree.query(coords, k=k + 1)
    print(f'  kNN graph built.  shape: {indices[:, 1:].shape}')
    return indices[:, 1:].astype(np.int64), distances[:, 1:].astype(np.float32)


def build_dummy_neighbors(n_cells, k=1):
    """Placeholder neighbour arrays when USE_KNN=False."""
    return (np.zeros((n_cells, k), dtype=np.int64),
            np.ones( (n_cells, k), dtype=np.float32))


def estimate_neighbor_cache_gb(n_cells, k, n_t, n_vars=3, dtype_bytes=4):
    return n_cells * k * n_t * n_vars * dtype_bytes / 1024**3

### 3.5 Mock data generator (for smoke testing)

In [ ]:
def generate_mock_data(n_cells=200, n_timesteps=100, seed=42):
    """
    Generate synthetic u-velocity data for testing without real CSV files.

    Returns
    -------
    data   : np.ndarray  shape (n_cells, n_timesteps, 1)
    coords : np.ndarray  shape (n_cells, 2)
    """
    rng    = np.random.default_rng(seed)
    coords = rng.uniform(0, 1, size=(n_cells, 2)).astype(np.float32)
    t      = np.linspace(0, 4 * np.pi, n_timesteps)
    data   = np.zeros((n_cells, n_timesteps, 1), dtype=np.float32)
    for c in range(n_cells):
        x, y  = coords[c]
        phase = 2 * np.pi * (x + y)
        data[c, :, 0] = np.cos(t + phase) + 0.05 * rng.standard_normal(n_timesteps)
    print(f'Mock data shape   : {data.shape}')
    print(f'Mock coords shape : {coords.shape}')
    return data, coords

### 3.6 Load and preprocess data

In [ ]:
print('=' * 60)
print('Loading data ...')
print('=' * 60)

if USE_MOCK_DATA:
    raw_data, coords = generate_mock_data(n_cells=300, n_timesteps=120)
else:
    raw_data, coords = load_csv_data_univariate(U_VELOCITY_CSV)

print('\nValidating data ...')
assert validate_data(raw_data), 'Data validation failed.'

print('\nNormalising (z-score, u-velocity) ...')
data_norm, u_scaler = normalize_data(raw_data)

print('\nBuilding feature tensor [u_norm, x_norm, y_norm] ...')
features = build_feature_tensor(data_norm, coords)

if USE_KNN:
    print(f'\nBuilding kNN graph (k={K_NEIGHBORS}) ...')
    neighbor_indices, neighbor_distances = build_knn_graph(coords, K_NEIGHBORS)
else:
    print('\nUSE_KNN=False — skipping spatial graph (using dummy neighbours).')
    neighbor_indices, neighbor_distances = build_dummy_neighbors(features.shape[0], k=1)

n_cells, n_timesteps, n_features = features.shape
print(f"\n{'='*60}")
print(f'Dataset summary')
print(f'  Cells      : {n_cells}')
print(f'  Timesteps  : {n_timesteps}')
print(f'  Features   : {n_features}  (u_norm, x_norm, y_norm)')
print(f"  kNN        : {'enabled (k=' + str(K_NEIGHBORS) + ')' if USE_KNN else 'disabled'}")
print(f"{'='*60}")

## 4 — SpatioTemporalDataset

In [ ]:
class SpatioTemporalDataset(Dataset):
    """
    Sliding-window dataset for univariate spatiotemporal forecasting.

    Each sample contains:
      - center_seq       : (seq_len, 3)      target cell features [u, x_norm, y_norm]
      - neighbor_seq     : (seq_len, k, 3)   k neighbours features
      - target           : (pred_steps, 1)   ground-truth future u values
      - neighbor_targets : (k, pred_steps, 1) future u for neighbours
      - neighbor_dists   : (k,)              spatial distances
    """

    def __init__(
        self,
        features,
        target_data,
        neighbor_indices,
        neighbor_distances,
        seq_length,
        pred_steps,
        start_t,
        end_t,
        device=None,
        preload_to_gpu=False,
        cache_neighbors=False,
    ):
        super().__init__()
        self.seq_length = seq_length
        self.pred_steps = pred_steps
        self.start_t    = start_t
        self.n_cells    = features.shape[0]

        feat_tensor = torch.FloatTensor(features)
        tgt_tensor  = torch.FloatTensor(target_data)
        nbr_tensor  = torch.LongTensor(neighbor_indices)
        dist_tensor = torch.FloatTensor(neighbor_distances)

        if preload_to_gpu and device is not None:
            feat_tensor = feat_tensor.to(device)
            tgt_tensor  = tgt_tensor.to(device)
            nbr_tensor  = nbr_tensor.to(device)
            dist_tensor = dist_tensor.to(device)

        self.features           = feat_tensor
        self.target_data        = tgt_tensor
        self.neighbor_indices   = nbr_tensor
        self.neighbor_distances = dist_tensor
        self.n_valid_windows    = max(0, end_t - start_t - seq_length - pred_steps + 1)

        self._use_cache = cache_neighbors
        if cache_neighbors:
            self.neighbor_feat_cache = self.features[self.neighbor_indices]
            self.neighbor_tgt_cache  = self.target_data[self.neighbor_indices]

    def __len__(self):
        return self.n_cells * self.n_valid_windows

    def __getitem__(self, idx):
        cell_idx   = idx // self.n_valid_windows
        window_idx = idx  % self.n_valid_windows
        t0 = self.start_t + window_idx
        t1 = t0 + self.seq_length

        center_seq = self.features[cell_idx, t0:t1, :]   # (L, 3)

        if self._use_cache:
            neighbor_seq     = self.neighbor_feat_cache[cell_idx, :, t0:t1, :]
            neighbor_targets = self.neighbor_tgt_cache[cell_idx, :, t1:t1 + self.pred_steps, :]
        else:
            nbr_idx          = self.neighbor_indices[cell_idx]
            neighbor_seq     = self.features[nbr_idx, t0:t1, :]            # (k, L, 3)
            neighbor_targets = self.target_data[nbr_idx, t1:t1 + self.pred_steps, :]  # (k,P,1)

        neighbor_seq = neighbor_seq.permute(1, 0, 2)              # (L, k, 3)
        target          = self.target_data[cell_idx, t1:t1 + self.pred_steps, :]  # (P, 1)
        neighbor_dists  = self.neighbor_distances[cell_idx]       # (k,)

        return center_seq, neighbor_seq, target, neighbor_targets, neighbor_dists

### 4.1 Build train / val / test DataLoaders

In [ ]:
def build_dataloaders(features, target_data, neighbor_indices,
                      neighbor_distances, config, device):
    n_t        = features.shape[1]
    seq_len    = config['INPUT_SEQ_LENGTH']
    pred_steps = config['PRED_FULL_HORIZON']
    train_end  = int(n_t * config['TRAIN_RATIO'])
    val_end    = int(n_t * (config['TRAIN_RATIO'] + config['VAL_RATIO']))

    shared = dict(
        features           = features,
        target_data        = target_data,
        neighbor_indices   = neighbor_indices,
        neighbor_distances = neighbor_distances,
        seq_length         = seq_len,
        pred_steps         = pred_steps,
        device             = device,
        preload_to_gpu     = config['PRELOAD_TO_GPU'],
        cache_neighbors    = config['CACHE_NEIGHBORS'],
    )

    print('Creating datasets ...')
    train_ds = SpatioTemporalDataset(start_t=0,         end_t=train_end, **shared)
    val_ds   = SpatioTemporalDataset(start_t=train_end, end_t=val_end,   **shared)
    test_ds  = SpatioTemporalDataset(start_t=val_end,   end_t=n_t,       **shared)

    dl_kwargs = dict(
        batch_size  = config['BATCH_SIZE'],
        num_workers = config['NUM_WORKERS'],
        pin_memory  = config['PIN_MEMORY'] and device.type == 'cuda',
    )
    print(f'  Train samples: {len(train_ds)}')
    print(f'  Val   samples: {len(val_ds)}')
    print(f'  Test  samples: {len(test_ds)}')

    return (
        DataLoader(train_ds, shuffle=True,  **dl_kwargs),
        DataLoader(val_ds,   shuffle=False, **dl_kwargs),
        DataLoader(test_ds,  shuffle=False, **dl_kwargs),
    )

## 5 — Model Architectures

Input features = **3** (`[u, x_norm, y_norm]`).  Output = **1** channel (`u`).

When `USE_KNN=True` the `DistanceWeightedSpatialEncoder` appends spatial context;
when `USE_KNN=False` the spatial encoder is bypassed entirely.

In [ ]:
# ------------------------------------------------------------------
# 5.1  Distance-Weighted Spatial Encoder  (used when USE_KNN=True)
# ------------------------------------------------------------------

class DistanceWeightedSpatialEncoder(nn.Module):
    """
    Encodes k-nearest-neighbour features into a spatial context embedding
    using inverse-distance-weighted aggregation.

    Parameters
    ----------
    input_features : int   features per cell (default: 3)
    embed_dim      : int   output embedding dimension
    """

    def __init__(self, input_features=3, embed_dim=64):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_features, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim),
        )
        self.dist_scale = nn.Sequential(
            nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1), nn.Softplus(),
        )

    def forward(self, neighbor_seq, neighbor_dists):
        # neighbor_seq   : (B, L, k, input_features)
        # neighbor_dists : (B, k)
        embedded = self.mlp(neighbor_seq)                          # (B, L, k, E)
        d        = neighbor_dists.unsqueeze(-1)                    # (B, k, 1)
        raw_w    = self.dist_scale(d)                              # (B, k, 1)
        inv_w    = 1.0 / (raw_w + 1e-6)
        weights  = inv_w / (inv_w.sum(dim=1, keepdim=True) + 1e-8)  # (B, k, 1)
        weights  = weights.unsqueeze(1)                            # (B, 1, k, 1)
        return (embedded * weights).sum(dim=2)                     # (B, L, E)


# ------------------------------------------------------------------
# 5.2  SpatioTemporalLSTM  (univariate)
# ------------------------------------------------------------------

class SpatioTemporalLSTM(nn.Module):
    """
    LSTM for univariate spatiotemporal forecasting.

    Input  : center_seq (B, L, 3) = [u, x_norm, y_norm]
    Output : (B, pred_steps, 1)  = predicted u

    When use_knn=True the spatial encoder is applied on k neighbours
    and its output is concatenated to center_seq before the LSTM.
    When use_knn=False only center_seq is used.
    """

    def __init__(self, input_features=3, spatial_embed_dim=64,
                 hidden_size=256, num_layers=3, pred_steps=5,
                 dropout=0.2, use_knn=False):
        super().__init__()
        self.pred_steps = pred_steps
        self.use_knn    = use_knn
        if use_knn:
            self.spatial_encoder = DistanceWeightedSpatialEncoder(
                input_features, spatial_embed_dim)
            lstm_input_dim = input_features + spatial_embed_dim
        else:
            lstm_input_dim = input_features
        self.lstm = nn.LSTM(
            input_size  = lstm_input_dim,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0,
        )
        self.decoder = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, pred_steps),
        )

    def forward(self, center_seq, neighbor_seq, neighbor_dists):
        if self.use_knn:
            spatial_embed = self.spatial_encoder(neighbor_seq, neighbor_dists)
            x = torch.cat([center_seq, spatial_embed], dim=-1)
        else:
            x = center_seq
        lstm_out, _ = self.lstm(x)
        out = self.decoder(lstm_out[:, -1, :])   # (B, pred_steps)
        return out.unsqueeze(-1)                  # (B, P, 1)


# ------------------------------------------------------------------
# 5.3  SpatioTemporalTransformer  (univariate)
# ------------------------------------------------------------------

class SpatioTemporalTransformer(nn.Module):
    """
    Transformer for univariate spatiotemporal forecasting.

    Input  : center_seq (B, L, 3) = [u, x_norm, y_norm]
    Output : (B, pred_steps, 1)  = predicted u

    When use_knn=True the spatial encoder is applied on k neighbours
    and its output is concatenated before the linear projection.
    When use_knn=False only center_seq is projected.
    """

    def __init__(self, input_features=3, spatial_embed_dim=64,
                 d_model=256, nhead=8, num_layers=3, pred_steps=5,
                 dropout=0.2, use_knn=False):
        super().__init__()
        assert d_model % nhead == 0
        self.pred_steps = pred_steps
        self.use_knn    = use_knn
        if use_knn:
            self.spatial_encoder = DistanceWeightedSpatialEncoder(
                input_features, spatial_embed_dim)
            proj_in = input_features + spatial_embed_dim
        else:
            proj_in = input_features
        self.input_projection = nn.Linear(proj_in, d_model)
        self.pos_encoding     = nn.Parameter(torch.randn(1, 512, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        self.decoder = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, pred_steps),
        )

    def forward(self, center_seq, neighbor_seq, neighbor_dists):
        if self.use_knn:
            spatial_embed = self.spatial_encoder(neighbor_seq, neighbor_dists)
            x = torch.cat([center_seq, spatial_embed], dim=-1)
        else:
            x = center_seq
        x = self.input_projection(x)
        L = x.shape[1]
        x = x + self.pos_encoding[:, :L, :]
        x = self.transformer_encoder(x)
        out = self.decoder(x[:, -1, :])   # (B, pred_steps)
        return out.unsqueeze(-1)           # (B, P, 1)


# ------------------------------------------------------------------
# 5.4  Model factory
# ------------------------------------------------------------------

def create_model(config):
    model_type = config['MODEL_TYPE']
    kwargs = dict(
        input_features    = 3,
        spatial_embed_dim = config['SPATIAL_EMBED_DIM'],
        num_layers        = config['NUM_LAYERS'],
        pred_steps        = config['PRED_FULL_HORIZON'],
        dropout           = config['DROPOUT'],
        use_knn           = config['USE_KNN'],
    )
    if model_type == 'SpatioTemporalLSTM':
        model = SpatioTemporalLSTM(hidden_size=config['HIDDEN_SIZE'], **kwargs)
    elif model_type == 'SpatioTemporalTransformer':
        model = SpatioTemporalTransformer(
            d_model=config['HIDDEN_SIZE'], nhead=config['NHEAD'], **kwargs)
    else:
        raise ValueError(f"Unknown MODEL_TYPE: '{model_type}'")
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Created {model_type}')
    print(f'  Trainable parameters : {n_params:,}')
    print(f"  USE_KNN              : {config['USE_KNN']}")
    return model

## 6 — Loss Functions

Primary loss: **MSE** on u-velocity.  Spatial smoothness is active only when `USE_KNN=True`.

In [ ]:
def spatial_smoothness_loss(predictions, neighbor_targets, neighbor_dists):
    # predictions      : (B, P, 1)
    # neighbor_targets : (B, k, P, 1)
    # neighbor_dists   : (B, k)
    diff_sq = (predictions.unsqueeze(1) - neighbor_targets) ** 2  # (B,k,P,1)
    inv_d   = 1.0 / (neighbor_dists + 1e-6)
    weights = inv_d / (inv_d.sum(dim=1, keepdim=True) + 1e-8)     # (B,k)
    weights = weights.unsqueeze(-1).unsqueeze(-1)                  # (B,k,1,1)
    return (diff_sq * weights).sum(dim=1).mean()


def compute_total_loss(predictions, targets, neighbor_targets,
                       neighbor_dists, lambda_smooth, use_knn):
    mse_val    = nn.functional.mse_loss(predictions, targets)
    smooth_val = (
        spatial_smoothness_loss(predictions, neighbor_targets, neighbor_dists)
        if use_knn else torch.tensor(0.0, device=predictions.device)
    )
    total = mse_val + lambda_smooth * smooth_val
    return total, mse_val.item(), smooth_val.item()

## 7 — Training Utilities

In [ ]:
def move_batch_to_device(batch, device):
    return tuple(t.to(device, non_blocking=True) for t in batch)


def train_step(model, batch, optimizer, scaler, config, primary_device, all_devices):
    center_seq, neighbor_seq, target, neighbor_targets, neighbor_dists = \
        move_batch_to_device(batch, primary_device)
    optimizer.zero_grad()
    with autocast(enabled=config['USE_AMP'] and primary_device.type == 'cuda'):
        preds = model(center_seq, neighbor_seq, neighbor_dists)
        loss, mse_val, smooth_val = compute_total_loss(
            preds, target, neighbor_targets, neighbor_dists,
            config['LAMBDA_SPATIAL_SMOOTH'], config['USE_KNN'],
        )
    if config['USE_AMP'] and primary_device.type == 'cuda':
        scaler.scale(loss).backward()
        if config['GRAD_CLIP'] > 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), config['GRAD_CLIP'])
        scaler.step(optimizer)
        scaler.update()
    else:
        loss.backward()
        if config['GRAD_CLIP'] > 0:
            nn.utils.clip_grad_norm_(model.parameters(), config['GRAD_CLIP'])
        optimizer.step()
    return loss.item(), mse_val, smooth_val

## 8 — Validation, Early Stopping, and Checkpointing

In [ ]:
@torch.no_grad()
def validate(model, val_loader, config, primary_device):
    model.eval()
    total_sum = mse_sum = smooth_sum = mae_sum = 0.0
    n = 0
    for batch in val_loader:
        c, nb, tgt, nb_tgt, nb_d = move_batch_to_device(batch, primary_device)
        with autocast(enabled=config['USE_AMP'] and primary_device.type == 'cuda'):
            preds = model(c, nb, nb_d)
            loss, mse_val, smooth_val = compute_total_loss(
                preds, tgt, nb_tgt, nb_d,
                config['LAMBDA_SPATIAL_SMOOTH'], config['USE_KNN'],
            )
        total_sum  += loss.item()
        mse_sum    += mse_val
        smooth_sum += smooth_val
        mae_sum    += (preds - tgt).abs().mean().item()
        n += 1
    if n == 0:
        return {'val_loss': 0.0, 'val_mse': 0.0, 'val_smooth': 0.0, 'val_mae': 0.0}
    return {
        'val_loss'   : total_sum  / n,
        'val_mse'    : mse_sum    / n,
        'val_smooth' : smooth_sum / n,
        'val_mae'    : mae_sum    / n,
    }


class EarlyStopping:
    def __init__(self, patience=10, min_delta=1e-5):
        self.patience    = patience
        self.min_delta   = min_delta
        self.best_loss   = float('inf')
        self.counter     = 0
        self.should_stop = False

    def step(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop


def save_checkpoint(model, optimizer, epoch, val_loss, path):
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    torch.save({'epoch': epoch, 'val_loss': val_loss,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict()}, path)

## 9 — Training Loop

In [ ]:
def log_hyperparameters(config):
    print('\n' + '=' * 60)
    print('Hyper-parameter log')
    print('=' * 60)
    for k, v in config.items():
        print(f'  {k:<30s} : {v}')
    print('=' * 60 + '\n')


def train_model(model, train_loader, val_loader, config, primary_device, all_devices):
    log_hyperparameters(config)
    optimizer  = optim.AdamW(model.parameters(),
                             lr=config['LEARNING_RATE'],
                             weight_decay=config['WEIGHT_DECAY'])
    scheduler  = optim.lr_scheduler.StepLR(optimizer,
                                            step_size=config['SCHEDULER_STEP'],
                                            gamma=config['SCHEDULER_GAMMA'])
    scaler     = GradScaler(enabled=config['USE_AMP'] and primary_device.type == 'cuda')
    early_stop = EarlyStopping(patience=config['PATIENCE'])
    best_ckpt  = os.path.join(config['CHECKPOINT_DIR'], 'best_model.pth')
    last_ckpt  = os.path.join(config['CHECKPOINT_DIR'], 'last_model.pth')

    history = {'train_loss': [], 'train_mse': [], 'train_smooth': [],
               'val_loss': [], 'val_mse': [], 'val_smooth': [],
               'val_mae': [], 'lr': []}
    best_val_loss = float('inf')

    print(f"Starting training for up to {config['EPOCHS']} epochs ...")
    for epoch in range(1, config['EPOCHS'] + 1):
        model.train()
        e_loss = e_mse = e_smooth = 0.0
        n_batches = 0
        pbar = tqdm(train_loader,
                    desc=f"Epoch {epoch:>3d}/{config['EPOCHS']} [Train]",
                    leave=False, dynamic_ncols=True)
        for batch in pbar:
            t_loss, t_mse, t_smooth = train_step(
                model, batch, optimizer, scaler,
                config, primary_device, all_devices)
            e_loss    += t_loss
            e_mse     += t_mse
            e_smooth  += t_smooth
            n_batches += 1
            pbar.set_postfix(loss=f'{t_loss:.4f}')

        if n_batches == 0:
            continue

        train_m = {'train_loss':   e_loss   / n_batches,
                   'train_mse':    e_mse    / n_batches,
                   'train_smooth': e_smooth / n_batches}
        val_m   = validate(model, val_loader, config, primary_device)
        scheduler.step()
        lr = optimizer.param_groups[0]['lr']

        for k, v in {**train_m, **val_m}.items():
            history[k].append(v)
        history['lr'].append(lr)

        print(f"Epoch {epoch:>3d}  "
              f"train_loss={train_m['train_loss']:.4f}  "
              f"val_loss={val_m['val_loss']:.4f}  "
              f"val_mae={val_m['val_mae']:.4f}  "
              f"lr={lr:.2e}")

        if val_m['val_loss'] < best_val_loss:
            best_val_loss = val_m['val_loss']
            save_checkpoint(model, optimizer, epoch, best_val_loss, best_ckpt)
            print(f'  => New best model saved (val_loss={best_val_loss:.4f})')

        save_checkpoint(model, optimizer, epoch, val_m['val_loss'], last_ckpt)

        if early_stop.step(val_m['val_loss']):
            print(f'Early stopping at epoch {epoch}.')
            break

    if os.path.exists(best_ckpt):
        ckpt = torch.load(best_ckpt, map_location=primary_device)
        model.load_state_dict(ckpt['model_state'])
        print(f"Loaded best model (epoch {ckpt['epoch']}, "
              f"val_loss={ckpt['val_loss']:.4f})")
    return model, history

## 10 — Inference & Evaluation

In [ ]:
@torch.no_grad()
def evaluate_on_loader(model, loader, device, use_amp=True):
    """Compute MAE and RMSE on the u-velocity channel."""
    model.eval()
    all_preds, all_targets = [], []
    for batch in tqdm(loader, desc='Evaluating', leave=False):
        c, nb, tgt, _, nb_d = move_batch_to_device(batch, device)
        with autocast(enabled=use_amp and device.type == 'cuda'):
            preds = model(c, nb, nb_d)
        all_preds.append(preds.cpu())
        all_targets.append(tgt.cpu())
    if not all_preds:
        return {'mae': float('nan'), 'rmse': float('nan')}
    preds_np   = torch.cat(all_preds,   dim=0).numpy()   # (N, P, 1)
    targets_np = torch.cat(all_targets, dim=0).numpy()   # (N, P, 1)
    mae  = float(np.abs(preds_np - targets_np).mean())
    rmse = float(np.sqrt(((preds_np - targets_np) ** 2).mean()))
    print(f'  MAE   (u_velocity): {mae:.6f}')
    print(f'  RMSE  (u_velocity): {rmse:.6f}')
    return {'mae': mae, 'rmse': rmse}

### 10.1 Training curve visualisation

In [ ]:
def plot_training_curves(history, output_dir):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot(epochs, history['train_loss'], label='Train loss', color='steelblue')
    axes[0].plot(epochs, history['val_loss'],   label='Val loss',   color='tomato',
                 linestyle='--')
    axes[0].set_title('Total Loss (MSE + Smoothness when USE_KNN=True)')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history['val_mae'], label='Val MAE', color='mediumseagreen')
    axes[1].set_title('Validation MAE')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MAE')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = os.path.join(output_dir, 'training_curves.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved training curves -> {save_path}')


def plot_predictions(model, test_loader, device, u_scaler,
                     output_dir, n_samples=4, use_amp=True):
    """Plot prediction vs ground truth for a few test samples."""
    model.eval()
    try:
        batch = next(iter(test_loader))
    except StopIteration:
        print('  Empty test loader — no samples to plot.')
        return
    n_samples = min(n_samples, batch[0].shape[0])
    c, nb, tgt, _, nb_d = move_batch_to_device(batch, device)
    with autocast(enabled=use_amp and device.type == 'cuda'):
        preds = model(c, nb, nb_d)
    preds_np  = preds[:n_samples].cpu().numpy()    # (n, P, 1)
    target_np = tgt[:n_samples].cpu().numpy()      # (n, P, 1)

    fig, axes = plt.subplots(1, n_samples, figsize=(4 * n_samples, 4))
    if n_samples == 1:
        axes = [axes]
    for s, ax in enumerate(axes):
        p_dn = u_scaler.inverse_transform(preds_np[s, :, 0:1]).flatten()
        t_dn = u_scaler.inverse_transform(target_np[s, :, 0:1]).flatten()
        steps = np.arange(1, len(p_dn) + 1)
        ax.plot(steps, t_dn, 'o-', label='Ground Truth', color='steelblue')
        ax.plot(steps, p_dn, 's--', label='Prediction',  color='tomato')
        ax.set_title(f'Sample {s+1} — U-velocity')
        ax.set_xlabel('Prediction Step')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    save_path = os.path.join(output_dir, 'prediction_examples.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved prediction examples -> {save_path}')

## 11 — Run Training

Assembles the config dict, creates the model, builds DataLoaders, and trains.

In [ ]:
config = {
    'USE_MULTI_GPU'       : USE_MULTI_GPU,
    'AUTO_SELECT_DEVICES' : AUTO_SELECT_DEVICES,
    'DEVICE_IDS'          : DEVICE_IDS,
    'MODEL_TYPE'          : MODEL_TYPE,
    'SPATIAL_EMBED_DIM'   : SPATIAL_EMBED_DIM,
    'HIDDEN_SIZE'         : HIDDEN_SIZE,
    'NUM_LAYERS'          : NUM_LAYERS,
    'NHEAD'               : NHEAD,
    'DROPOUT'             : DROPOUT,
    'INPUT_SEQ_LENGTH'    : INPUT_SEQ_LENGTH,
    'PRED_FULL_HORIZON'   : PRED_FULL_HORIZON,
    'K_NEIGHBORS'         : K_NEIGHBORS,
    'USE_KNN'             : USE_KNN,
    'BATCH_SIZE'          : BATCH_SIZE,
    'LEARNING_RATE'       : LEARNING_RATE,
    'WEIGHT_DECAY'        : WEIGHT_DECAY,
    'EPOCHS'              : EPOCHS,
    'PATIENCE'            : PATIENCE,
    'GRAD_CLIP'           : GRAD_CLIP,
    'SCHEDULER_STEP'      : SCHEDULER_STEP,
    'SCHEDULER_GAMMA'     : SCHEDULER_GAMMA,
    'LAMBDA_SPATIAL_SMOOTH': LAMBDA_SPATIAL_SMOOTH,
    'TRAIN_RATIO'         : TRAIN_RATIO,
    'VAL_RATIO'           : VAL_RATIO,
    'USE_AMP'             : USE_AMP,
    'PRELOAD_TO_GPU'      : PRELOAD_TO_GPU,
    'CACHE_NEIGHBORS'     : CACHE_NEIGHBORS,
    'CACHE_VRAM_FRACTION' : CACHE_VRAM_FRACTION,
    'NUM_WORKERS'         : NUM_WORKERS,
    'PIN_MEMORY'          : PIN_MEMORY,
    'OUTPUT_DIR'          : OUTPUT_DIR,
    'CHECKPOINT_DIR'      : CHECKPOINT_DIR,
}

train_loader, val_loader, test_loader = build_dataloaders(
    features, data_norm, neighbor_indices, neighbor_distances,
    config, PRIMARY_DEVICE
)

model = create_model(config).to(PRIMARY_DEVICE)

model, history = train_model(
    model, train_loader, val_loader, config, PRIMARY_DEVICE, ALL_DEVICES
)

## 12 — Visualise & Evaluate

In [ ]:
plot_training_curves(history, OUTPUT_DIR)

print('\n' + '=' * 60)
print('Test-set evaluation')
print('=' * 60)
test_metrics = evaluate_on_loader(model, test_loader, PRIMARY_DEVICE, USE_AMP)

print('\nPlotting prediction examples ...')
plot_predictions(
    model, test_loader, PRIMARY_DEVICE,
    u_scaler, OUTPUT_DIR,
    n_samples=4, use_amp=USE_AMP,
)

print('\nAll done!')
print(f'Checkpoints saved in   : {CHECKPOINT_DIR}/')
print(f'Visualisations saved in: {OUTPUT_DIR}/')